# Simple U-Net Denoiser (CIFAR-10, 64x64, single objective)

Trains `TinyUNet64` (`code/unet_tiny_denoising_64.py`) with the
standard `"x0"` objective only -- the network directly predicts the
clean image, `x_hat = UNet(x_noisy)`. No second model, no objective
comparison: this is the simplest possible "how well does this U-Net
denoise?" setup. (For the x0-vs-noise-objective comparison, see
`code/compare_denoising_objectives.py` and its notebook instead.)

Task: a clean image `x` is corrupted with additive Gaussian noise,
`x_noisy = x + sigma*eps`, `eps ~ N(0,I)`, `sigma = noise_std`, and the
U-Net learns `x_noisy -> x_hat ~= x` via `L = ||x - x_hat||^2`.

Because there is only one objective here, the **training loss** and the
**test-set denoising MSE** are the exact same formula -- `MSE(x_hat, x)`
-- computed on different data splits:

| | train_loss | test_denoising_mse |
|---|---|---|
| data | training images | held-out test images |
| model mode | `model.train()`, gradients on | `model.eval()`, `no_grad()` |
| parameters | changing mid-epoch (mix of stale/updated) | one frozen snapshot |
| meaning | optimization-progress signal | the actual generalization/denoising-quality number |

Core code lives in `code/unet_denoising_simple.py` (version-controlled);
this notebook is a thin orchestration layer: GPU check, repo pull,
build model + print the full shape trace, download real CIFAR-10
resized to 64x64 (cached in Drive), train, plot the loss curve, and
show clean/noisy/denoised qualitative + quantitative results as the
required results-visualization step.

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

Model code lives in `code/unet_tiny_denoising_64.py` (architecture) and
`code/unet_denoising_simple.py` (training/eval loop, visualization),
not duplicated in this notebook. Replace `REPO_URL` with this
repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 3. Build model, print full shape trace

One verbose forward pass on a small synthetic batch, purely to print
the `[B,C,H,W]` shape before and after every op -- including every
skip-connection save and every `torch.cat`. Not part of training.

In [ ]:
from unet_tiny_denoising_64 import TinyUNet64, count_trainable_parameters

torch.manual_seed(0)

BASE_CHANNELS = 32

model = TinyUNet64(in_channels=3, out_channels=3, base_channels=BASE_CHANNELS, prediction_mode="x0").to(device)
print(f"trainable parameters: {count_trainable_parameters(model):,}")

sample_batch = torch.rand(4, 3, 64, 64, device=device)  # [B,3,64,64], synthetic, in [0,1]
_ = model(sample_batch, verbose=True)


## 4. Download real CIFAR-10 (train/test split), resized to 64x64, cached in Drive

CIFAR-10 is natively 32x32; `get_cifar10_denoising_dataloaders` resizes
every image up to 64x64 so it matches the resolution `TinyUNet64` was
designed for. Labels are unused (the denoising target is the clean
image itself).

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so we mount Google Drive and point `DATA_ROOT` at a
persistent Drive folder: `torchvision` downloads into it once and
reads from it on every later run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Persistent cache: downloaded once into Drive, reused on every
# future Colab session (local /content storage is ephemeral).
DATA_ROOT = "/content/drive/MyDrive/thesis_data/cifar10"


## 5. Train

Explicit train/eval loops (`train_one_epoch`, `evaluate`, both defined
in `code/unet_denoising_simple.py`): each clean batch is corrupted with
additive Gaussian noise (`noise_std`) and the U-Net is trained to
recover the clean image, MSE loss, Adam optimizer.

In [ ]:
import torch.nn as nn
from unet_denoising_simple import train_one_epoch, evaluate
from unet_tiny_denoising_64 import get_cifar10_denoising_dataloaders

BATCH_SIZE = 128
IMAGE_SIZE = 64
N_EPOCHS = 10
LR = 1e-3
NOISE_STD = 0.3  # std of additive Gaussian noise defining the denoising task

train_loader, test_loader = get_cifar10_denoising_dataloaders(DATA_ROOT, BATCH_SIZE, image_size=IMAGE_SIZE)
print(f"train batches = {len(train_loader)}, test batches = {len(test_loader)}")

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_loss": [], "test_denoising_mse": []}

for epoch in range(N_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device, NOISE_STD)
    test_mse = evaluate(model, test_loader, loss_fn, device, NOISE_STD)

    history["train_loss"].append(train_loss)
    history["test_denoising_mse"].append(test_mse)

    print(f"epoch {epoch + 1:3d}/{N_EPOCHS}  train_loss={train_loss:.4f}  test_denoising_mse={test_mse:.4f}")


## 6. Loss curve (quantitative result over training)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_loss"], label="train_loss")
ax.plot(history["test_denoising_mse"], label="test_denoising_mse")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE(x_hat, clean)")
ax.set_title("Denoising loss")
ax.legend()
fig.tight_layout()
plt.show()

print(f"final test_denoising_mse = {history['test_denoising_mse'][-1]:.4f}")


## 7. Qualitative + quantitative check: clean vs. noisy vs. denoised

Per-project convention: always look at concrete examples, not only
scalar metrics. Shows 5 real test images as clean (row 1) / noisy input
(row 2) / U-Net output (row 3), with per-image MSE and PSNR (output vs.
clean) printed under each denoised image.

In [ ]:
from unet_tiny_denoising_64 import visualize_denoising

sample_images, _sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

visualize_denoising(model, sample_images, NOISE_STD, num_examples=5)


## 8. Final test-set denoising MSE

In [ ]:
final_test_mse = evaluate(model, test_loader, loss_fn, device, NOISE_STD)
print(f"average test-set denoising MSE (noise_std={NOISE_STD}): {final_test_mse:.4f}")
